### CAPA BRONZE — Ingesta Raw del TMDB 5000 Movie Dataset
##### Notebook: 01_bronze_movies_raw
##### Input:  Tabla gestionada en Unity Catalog
##### Output: streaming_clustering.bronze_movies_raw (Delta Table)

In [0]:
from pyspark.sql import functions as F

######1. CONFIGURACIÓN INICIAL

In [0]:
SOURCE_TABLE = "default.tmdb_5000_movies"  # tu tabla en Unity Catalog
DATABASE_NAME = "streaming_clustering"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")
spark.sql(f"USE {DATABASE_NAME}")
print(f"✓ Usando base de datos: {DATABASE_NAME}")

✓ Usando base de datos: streaming_clustering


######2. VERIFICAR QUE LA TABLA EXISTE

In [0]:
try:
    spark.sql(f"DESCRIBE TABLE {SOURCE_TABLE}")
    print(f"✓ Tabla encontrada: {SOURCE_TABLE}")
except Exception as e:
    print(f"✗ Tabla NO encontrada: {e}")
    raise

✓ Tabla encontrada: default.tmdb_5000_movies


######3. LEER LA TABLA CRUDA 

In [0]:
df_raw = spark.table(SOURCE_TABLE)

print(f"\n✓ Tabla leída correctamente")
print(f"  Filas   : {df_raw.count()}")
print(f"  Columnas: {len(df_raw.columns)}")
print(f"  Columnas: {df_raw.columns}")


✓ Tabla leída correctamente
  Filas   : 4806
  Columnas: 20
  Columnas: ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']


###### 4. AGREGAR METADATOS DE AUDITORÍA 
- _source_file     → de dónde vino el dato (trazabilidad)
- _ingestion_time  → cuándo fue ingerido (linaje temporal)

In [0]:
df_bronze = df_raw.withColumns({
    "_source_file":       F.lit(SOURCE_TABLE),       # nombre de la tabla origen
    "_ingestion_time":    F.current_timestamp(),
    "_ingestion_date":    F.current_date(),
    "_pipeline_version":  F.lit("1.0.0")
})

print("\n✓ Metadatos de auditoría añadidos:")
df_bronze.select(
    "_source_file", "_ingestion_time", "_ingestion_date", "_pipeline_version"
).show(3, truncate=60)


✓ Metadatos de auditoría añadidos:
+------------------------+--------------------------+---------------+-----------------+
|            _source_file|           _ingestion_time|_ingestion_date|_pipeline_version|
+------------------------+--------------------------+---------------+-----------------+
|default.tmdb_5000_movies|2026-05-01 23:01:09.508352|     2026-05-01|            1.0.0|
|default.tmdb_5000_movies|2026-05-01 23:01:09.508352|     2026-05-01|            1.0.0|
|default.tmdb_5000_movies|2026-05-01 23:01:09.508352|     2026-05-01|            1.0.0|
+------------------------+--------------------------+---------------+-----------------+
only showing top 3 rows


######5. PREVIEW DE LOS DATOS CRUDO

In [0]:
print("\n── Preview de datos crudos (Bronze) ──")
df_bronze.select(
    "id", "title", "budget", "genres",
    "keywords", "popularity", "vote_average",
    "_ingestion_time"
).show(5, truncate=50)

df_bronze.printSchema()



── Preview de datos crudos (Bronze) ──
+------+----------------------------------------+---------+--------------------------------------------------+--------------------------------------------------+----------+------------+-------------------------+
|    id|                                   title|   budget|                                            genres|                                          keywords|popularity|vote_average|          _ingestion_time|
+------+----------------------------------------+---------+--------------------------------------------------+--------------------------------------------------+----------+------------+-------------------------+
| 19995|                                  Avatar|237000000|[{"id": 28, "name": "Action"}, {"id": 12, "name...|[{"id": 1463, "name": "culture clash"}, {"id": ...|150.437577|         7.2|2026-05-01 23:01:10.75087|
|   285|Pirates of the Caribbean: At World's End|300000000|[{"id": 12, "name": "Adventure"}, {"id": 14, "n...|[{

######6. VALIDACIÓN BÁSICA DE CALIDAD 
- Solo contamos nulos — NO los corregimos (eso es tarea de Silver)

In [0]:
print("\n── Reporte de nulos por columna (Bronze no los corrige) ──")
null_counts = []
for c in df_raw.columns:
    n = df_bronze.filter(F.col(c).isNull()).count()
    if n > 0:
        null_counts.append((c, n))

if null_counts:
    for col_name, cnt in sorted(null_counts, key=lambda x: -x[1]):
        print(f"  {col_name}: {cnt} nulos")
else:
    print("  Sin nulos detectados en Bronze")


── Reporte de nulos por columna (Bronze no los corrige) ──
  homepage: 3091 nulos
  tagline: 848 nulos
  release_date: 7 nulos
  runtime: 7 nulos
  revenue: 6 nulos
  vote_count: 6 nulos
  title: 5 nulos
  vote_average: 5 nulos
  popularity: 4 nulos
  budget: 3 nulos
  id: 3 nulos
  overview: 3 nulos
  production_companies: 3 nulos
  production_countries: 3 nulos
  spoken_languages: 3 nulos
  status: 3 nulos


######7. GUARDAR COMO TABLA DELTA

In [0]:
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_ingestion_date")
    .saveAsTable("bronze_movies_raw")  # guardará en streaming_clustering.bronze_movies_raw
)
print("\n✓ Tabla bronze_movies_raw guardada como Delta Table")

# ── 8. VERIFICACIÓN FINAL ─────────────────────────────────────
df_ver = spark.table("bronze_movies_raw")
print(f"\n── Verificación final ──")
print(f"  Total filas    : {df_ver.count()}")
print(f"  Total columnas : {len(df_ver.columns)}")
print(f"  Columnas audit : {[c for c in df_ver.columns if c.startswith('_')]}")

spark.sql("SHOW TABLES IN streaming_clustering").show()

print("\nCapa Bronze completada.")
print(f"   Tabla: {DATABASE_NAME}.bronze_movies_raw")
print(f"   Avisa a los compañeros de Silver que pueden arrancar.")


✓ Tabla bronze_movies_raw guardada como Delta Table

── Verificación final ──
  Total filas    : 4806
  Total columnas : 24
  Columnas audit : ['_source_file', '_ingestion_time', '_ingestion_date', '_pipeline_version']
+--------------------+-----------------+-----------+
|            database|        tableName|isTemporary|
+--------------------+-----------------+-----------+
|streaming_clustering|bronze_movies_raw|      false|
+--------------------+-----------------+-----------+


Capa Bronze completada.
   Tabla: streaming_clustering.bronze_movies_raw
   Avisa a los compañeros de Silver que pueden arrancar.
